# Native discrete quantum walks in one and two dimensions

Preserve all original sizes and twenty-step evolutions. Complete states include the entangled coin environment; only compiler work must be zero. Position observations marginalize the coin. Classical step counts bind at synthesis time, so the two ring step counts are synthesized separately. Reusable native definitions live in workflow.py and the retained parent example. No classical simulation is substituted for a native quantum result.

## Keep the quantum walk coherent across steps

The visible roots compose the coin and reversible movement, with six position
bits on the ring or two five-bit coordinates on the torus. Static `t` expands
the repeated quantum program; it is not a measured stopping condition. The
coin is dropped only from the returned view. It may remain entangled with
position, so it must not be described as cleaned workspace.

Try a different admitted step count while retaining the original twenty-step
runs as the baseline. Compare the complete state before the sampled marginal:
1000 draws cannot detect a phase error. Ring wraparound and torus x/y ordering
are part of the quantum semantics. Compiler workspace leakage is checked
separately from the retained logical coin.


## Local native setup

Original code cell 2.

In [ ]:
from io import BytesIO
from typing import Annotated

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk_2d.discrete_time_quantum_walk.operations import (
    walk_step,
)
from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk_2d.discrete_time_quantum_walk.overview.workflow import (
    sample,
    verify_program,
)
from examples.synthesis.tutorials.advanced_tutorials.discrete_quantum_walk_2d.discrete_time_quantum_walk.physics import (
    COIN_AMPLITUDES,
    POSITION_AMPLITUDES,
)
from flagquantum import synth
from flagquantum.drawer import draw
from flagquantum.synth import (
    H,
    Output,
    QNum,
    S,
    allocate,
    control,
    drop,
    inplace_prepare_amplitudes,
    qfunc,
)


## Original one-dimensional size

Original code cell 3.

In [ ]:
POSITION_BITS = 6
CIRCLE_SIZE = 2**POSITION_BITS
NUM_SHOTS = 1000
print("Original ring:", CIRCLE_SIZE, "positions;", NUM_SHOTS, "actual shots")


## Native ring preparation and helpers

Original code cell 4.

In [ ]:
@qfunc
def increment(position: QNum) -> None:
    position += 1


@qfunc
def decrement(position: QNum) -> None:
    position -= 1


@qfunc
def ring(t: int, x: Output[Annotated[QNum, 6]]) -> None:
    coin: Annotated[QNum, 1] = QNum()
    allocate(x)
    allocate(coin)
    x ^= 32
    H(coin[0])
    S(coin[0])
    for step in range(t):
        H(coin[0])
        control(coin == 1, lambda: increment(x))
        control(coin == 0, lambda: decrement(x))
    drop(coin)


## One-step ring synthesis and full-state check

Original code cell 5.

In [ ]:
one_step = verify_program(
    1,
    1,
    synth.synthesize(ring, classical_inputs={"t": 1}, memory_limit_bytes=64 * 1024**2),
)
print(draw(one_step.program.circuit))
print("Absolute full-state error:", one_step.maximum_error)


## One-step actual observations

Original code cell 6.

In [ ]:
ring_one_observations = sample(one_step.program, shots=NUM_SHOTS)
print(ring_one_observations)


## Twenty-step actual observations and ring plot

Original code cell 7.

In [ ]:
long_ring = verify_program(
    1,
    20,
    synth.synthesize(ring, classical_inputs={"t": 20}, memory_limit_bytes=64 * 1024**2),
)
ring_observations = sample(long_ring.program, shots=NUM_SHOTS)
print(ring_observations)
figure, axis = plt.subplots(figsize=(8, 4))
axis.bar(
    [row.position[0] for row in ring_observations],
    [row.probability for row in ring_observations],
    width=0.5,
)
axis.set(
    xlim=(-0.5, CIRCLE_SIZE + 0.5),
    xlabel="Position",
    ylabel="Observed probability",
    title="64-position quantum ring: 20 steps, 1,000 local draws",
)
figure.tight_layout()
buffer = BytesIO()
figure.savefig(buffer, format="png", dpi=130)
display(Image(buffer.getvalue()))
plt.close(figure)


## Original two-dimensional size

Original code cell 9.

In [ ]:
POSITION_BITS = 5
CIRCLE_SIZE = 2**POSITION_BITS
NUM_SHOTS = 1000
print("Original torus:", CIRCLE_SIZE, "by", CIRCLE_SIZE)


## Reused native torus preparation and helpers

Original code cell 10.

In [ ]:
@qfunc
def walk_torus(
    t: int,
    x_amplitudes: list[float],
    y_amplitudes: list[float],
    coin_amplitudes: list[float],
    x: Output[Annotated[QNum, 5]],
    y: Output[Annotated[QNum, 5]],
) -> None:
    coin: Annotated[QNum, 2] = QNum()
    allocate(x)
    allocate(y)
    allocate(coin)
    inplace_prepare_amplitudes(x_amplitudes, 0, x)
    inplace_prepare_amplitudes(y_amplitudes, 0, y)
    inplace_prepare_amplitudes(coin_amplitudes, 0, coin)
    for step in range(t):
        walk_step(x, y, coin)
    drop(coin)


## Twenty-step torus synthesis and full-state check

Original code cell 11.

In [ ]:
torus = verify_program(
    2,
    20,
    synth.synthesize(
        walk_torus,
        classical_inputs={
            "t": 20,
            "x_amplitudes": list(POSITION_AMPLITUDES),
            "y_amplitudes": list(POSITION_AMPLITUDES),
            "coin_amplitudes": list(COIN_AMPLITUDES),
        },
        memory_limit_bytes=64 * 1024**2,
    ),
)
print("Actual resources:", torus.program.report["resources"])
print("Absolute full-state error:", torus.maximum_error)
print("Dirty compiler work:", torus.dirty_workspace_probability)


## Actual observations and probability heatmap

Original code cell 12.

In [ ]:
torus_observations = sample(torus.program, shots=NUM_SHOTS)
print(torus_observations)
Z = np.zeros((CIRCLE_SIZE, CIRCLE_SIZE), dtype=float)
for row in torus_observations:
    x, y = row.position
    Z[y, x] = row.probability
figure, axis = plt.subplots(figsize=(7, 6))
image = axis.imshow(
    Z,
    origin="lower",
    extent=(-0.5, CIRCLE_SIZE - 0.5, -0.5, CIRCLE_SIZE - 0.5),
    aspect="equal",
    interpolation="nearest",
    vmin=0,
    vmax=Z.max(),
)
figure.colorbar(image, ax=axis, label="Observed probability")
axis.set(
    xlabel="x", ylabel="y", title="32 x 32 quantum torus: 20 steps, 1,000 local draws"
)
figure.tight_layout()
buffer = BytesIO()
figure.savefig(buffer, format="png", dpi=130)
display(Image(buffer.getvalue()))
plt.close(figure)
